# Обучение baseline



## Создание или восстановление эксперимента

Выбери один из трёх сценариев:

- **Новый запуск:** `Experiment.setup(CONFIG_PATH, device=DEVICE)`.
- **Новый запуск с весами:** тот же `setup`, но с `weights_from="experiment_runs/previous/checkpoints/best.pt"`.
- **Продолжение:** `Experiment.resume(RESUME_RUN, device=DEVICE)`.

Имя нового запуска задаётся в YAML; существующая папка вызывает ошибку.
При продолжении используются конфиг, токенайзер и выборки сохранённого запуска.
`DEVICE = "auto"` выбирает CUDA, MPS или CPU; устройство в конфиге не хранится.

`RESUME_MAX_TOKENS` и `RESUME_MINI_EPOCH` можно оставить равными `None`.
При смене бюджета токенов текущая полная эпоха начинается заново;
веса, оптимизатор, scheduler и история сохраняются. Часть примеров повторится.
При смене интервала мини-эпох следующий интервал считается от точки восстановления.
Scheduler сохраняет исходное расписание по числу шагов.


In [ ]:
import sys
from pathlib import Path

project_dir = Path.cwd()
if not (project_dir / "lib").is_dir():
    project_dir = project_dir.parent
if not (project_dir / "lib").is_dir():
    raise FileNotFoundError("Открой ноутбук из корня HW-2 или папки notebooks/.")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

import matplotlib.pyplot as plt
import torch
from IPython.display import clear_output
from torch.nn import functional as F

from lib.train import train_loop
from lib.experiment import Experiment
from lib.utils.metrics import format_metrics, plot_losses, summarize_epoch
from lib.utils.experiment.checkpoint import capture_rng_state
from lib.utils.experiment.resume import restore_rng_state
from lib.utils.paths import PROJECT_DIR


In [ ]:
CONFIG_PATH = PROJECT_DIR / "experiment_configs" / "baseline_transformer.yaml"
DEVICE = "auto"
WEIGHTS_FROM = None
RESUME_RUN = None  # For example: PROJECT_DIR / "experiment_runs" / "baseline".
RESUME_MAX_TOKENS = None
RESUME_MINI_EPOCH = None

if RESUME_RUN is None:
    experiment = Experiment.setup(
        CONFIG_PATH, weights_from=WEIGHTS_FROM, device=DEVICE,
        details={"notes": "Baseline Transformer"},
    )
else:
    experiment = Experiment.resume(
        RESUME_RUN, device=DEVICE,
        max_tokens=RESUME_MAX_TOKENS,
        examples_per_mini_epoch=RESUME_MINI_EPOCH,
    )

print(f"Run: {experiment.run_dir}")
print(f"Device: {experiment.device}")
print(f"Completed epochs: {experiment.completed_epochs}")
print(f"Resume progress: {experiment.progress}")


## 2. Токенайзер

Токенайзер уже загружен в `experiment.tokenizer`.
При новом запуске используется файл из конфига, при resume — копия из `init_state/`.


In [ ]:
print(f"Vocabulary: {experiment.tokenizer.get_vocab_size():,}")
print(f"pad_id: {experiment.pad_id}")


## 3. Выборки

Выборки уже подготовлены по `data.train_size` и `data.val_size`.
При resume используются сохранённые индексы из `init_state/sample_indices.json`.
Повторное выполнение этой секции ничего не пересэмплирует.


In [ ]:
print(f"Train examples: {len(experiment.train_loader.dataset):,}")
print(f"Validation examples: {len(experiment.val_loader.dataset):,}")


## 4. Загрузчики батчей

Загрузчики доступны как `experiment.train_loader` и `experiment.val_loader`.


In [ ]:
print(f"Train batches: {len(experiment.train_loader):,}")
print(f"Validation batches: {len(experiment.val_loader):,}")


## 5. Модель, оптимизатор и scheduler

Объекты уже собраны. При resume восстанавливаются их сохранённые состояния.
`scheduler: null` отключает расписание; специального режима смены LR нет.


In [ ]:
num_parameters = sum(parameter.numel() for parameter in experiment.model.parameters())
print(f"Model: {type(experiment.model).__name__}, parameters: {num_parameters:,}")
print(f"Optimizer: {type(experiment.optimizer).__name__}")
print(f"Learning rates: {[group['lr'] for group in experiment.optimizer.param_groups]}")
scheduler_name = type(experiment.scheduler).__name__ if experiment.scheduler is not None else "disabled"
print(f"Scheduler: {scheduler_name}")


## 6. Просмотр одного батча без обучения

Эту секцию можно пропустить. Проверяем формы logits и значение loss без обновления весов.
После просмотра возвращаем загрузчик, режим модели и генератор случайных чисел
к состоянию перед просмотром, в том числе при resume.


In [ ]:
rng_state = capture_rng_state(experiment.device)
model_was_training = experiment.model.training

try:
    experiment.train_loader.set_epoch(experiment.completed_epochs)
    batch = next(iter(experiment.train_loader))
    batch = {name: tensor.to(experiment.device) for name, tensor in batch.items()}
    experiment.model.eval()
    with torch.no_grad():
        logits = experiment.model(
            encoder_input_ids=batch["encoder_input_ids"],
            decoder_input_ids=batch["decoder_input_ids"],
            encoder_key_padding_mask=batch["encoder_key_padding_mask"],
            decoder_key_padding_mask=batch["decoder_key_padding_mask"],
        )
        vocab_size = experiment.tokenizer.get_vocab_size()
        assert logits.shape == (*batch["label_ids"].shape, vocab_size)
        batch_loss = F.cross_entropy(
            logits.reshape(-1, vocab_size),
            batch["label_ids"].reshape(-1),
            ignore_index=experiment.pad_id,
        )
        assert torch.isfinite(batch_loss), "Batch loss is not finite."
    print(f"Logits shape: {tuple(logits.shape)}")
    print(f"Batch loss: {batch_loss.item():.4f}")
finally:
    experiment.model.train(model_was_training)
    experiment.train_loader.set_epoch(experiment.completed_epochs)
    restore_rng_state(rng_state, experiment.device)

del batch, logits, batch_loss

## Обучение и метрики

Ячейка ниже только определяет функции; обучение запускается отдельной ячейкой.
Цикл использует `experiment.progress`: при обычном resume уже обработанные
батчи пропускаются, при смене бюджета токенов эпоха начинается заново.

После мини-эпохи выполняются validation, шаг scheduler и один `save_checkpoint()`.
Он обновляет `last.pt`, при улучшении — `best.pt`, историю и краткую сводку.
Scheduler делает один шаг после каждой мини-эпохи внутри `train_loop`;
ноутбук отдельно его не вызывает.

Показываются эпоха, мини-эпоха, train/validation loss, время и секунды обучения
на пример. Время валидации, записи файлов и отрисовки в скорость обучения не входит.
График строит `lib.utils.metrics`; `show_metrics` можно заменить другим callback отображения.


In [ ]:
def show_metrics(experiment):
    clear_output(wait=True)
    print(format_metrics(experiment.validation_history[-1]))
    fig = plot_losses(experiment)
    plt.show()
    plt.close(fig)


def train_model(experiment):
    """The training loop stays here; Experiment only builds and saves state."""
    training = experiment.config["training"]
    mini_size = training["examples_per_mini_epoch"]

    def on_validation(metrics):
        experiment.record_validation({"epoch": epoch + 1, **metrics})
        epoch_finished = metrics["batch_index"] == len(experiment.train_loader)
        if epoch_finished:
            experiment.history.append(summarize_epoch(experiment.validation_history, epoch + 1))
        experiment.save_checkpoint()
        show_metrics(experiment)

    print(f"Completed epochs: {experiment.completed_epochs}/{training['num_epochs']}")
    for epoch in range(experiment.completed_epochs, training["num_epochs"]):
        experiment.train_loader.set_epoch(epoch)
        train_loop(
            model=experiment.model,
            train_loader=experiment.train_loader,
            optimizer=experiment.optimizer,
            device=experiment.device,
            pad_id=experiment.pad_id,
            val_loader=experiment.val_loader,
            examples_per_mini_epoch=mini_size,
            scheduler=experiment.scheduler,
            on_validation=on_validation,
            **experiment.progress,
        )


In [ ]:
train_model(experiment)


## Сводка

`results/summary.json` уже обновляется вместе с checkpoint. Эта необязательная
ячейка добавляет статус и сохраняет график; веса она не меняет.
После прерывания сначала сделай resume, чтобы сводка и график соответствовали
сохранённому состоянию, а не незавершённой работе в памяти.


In [ ]:
# Checkpoints already contain progress, and summary.json is updated on save.
experiment.save_results(details={
    "status": "completed" if experiment.completed_epochs >= experiment.config["training"]["num_epochs"] else "interrupted",
})

if experiment.history or experiment.validation_history:
    fig = plot_losses(experiment)
    fig.savefig(experiment.run_dir / "results" / "loss.png")
    plt.close(fig)

print(f"Completed epochs: {experiment.completed_epochs}")
print(f"Saved mini-epochs: {len(experiment.validation_history)}")
print(f"Best validation loss: {experiment.best_val_loss}")
print(f"Results: {experiment.run_dir / 'results'}")
